# MIMIC-IV 승압제 코호트 생성 가이드



## 1. 개요
MIMIC-IV 기반 의료 데이터 분석을 위한 전처리 파이프라인. 환자의 ICU 재실 기간을 1시간 단위의 고정 윈도우로 분할하고, 각 시간대별 승압제 투여 지속시간(Duration)을 산출함.

## 2. 처리 단계 (Pipeline)
1. **Windowing**: `icustays`의 입퇴실 시간을 기준으로 환자별 1시간 단위 타임라인 생성 (`windows_1h`).
2. **Filtering**: 분석 대상 승압제 4종(Norepi, Vaso, Epi, Dopa) 이벤트만 추출.
3. **Overlapping**: 각 타임라인 윈도우에 약물 투여 기간이 얼마나 겹치는지 계산 (단위: Hour, 0.0~1.0).
4. **Aggregation**: `stay_id`와 `window_start`를 기준으로 피벗하여 와이드 포맷(Wide Format) 데이터셋 생성.

## 3. 주요 컬럼 정보
| 컬럼명 | 설명 | 데이터 타입 |
| :--- | :--- | :--- |
| `stay_id` | ICU 입실 고유 ID | Integer |
| `window_start` | 1시간 구간 시작 시점 | Timestamp |
| `dur_norepi` | 노르에피네프린 투여 시간 (최대 1.0) | Float |
| `dur_vaso` | 바소프레신 투여 시간 (최대 1.0) | Float |
| `dur_epi` | 에피네프린 투여 시간 (최대 1.0) | Float |
| `dur_dopa` | 도파민 투여 시간 (최대 1.0) | Float |

## 4. 분석 가이드 (XGBoost/LightGBM 활용 시)
- **Sparsity**: 도파민(Dopa)과 에피네프린(Epi)은 데이터가 매우 적을 수 있으므로 사용 전 `df.describe()`로 확인 필수.
- **Leakage**: 특정 시점의 상태를 예측할 때, 해당 시점의 `dur_xxx` 변수를 피처로 쓸지 라벨로 쓸지 연구 설계에 따라 결정해야 함.

In [10]:
import duckdb
import pandas as pd
from pathlib import Path

# 1. DuckDB 연결 (제공해주신 경로 활용)
db_path = "/home/oracle/Coding/wsl_projects/miniprj/data-pipeline/data/duckdb/mimic_total.duckdb"
con = duckdb.connect(db_path)

In [12]:
# 2. 1시간 단위 슬라이딩 윈도우(windows_1h) 생성
# 기존에 생성된 CSV가 있다면 이를 활용하고, 없다면 icustays에서 직접 생성합니다.
window_csv_path = "/home/oracle/Coding/wsl_projects/miniprj/data-pipeline/data/processed/cohort_sliding_window_v2.csv"

if Path(window_csv_path).exists():
    con.execute(f"CREATE OR REPLACE TEMP VIEW windows_1h AS SELECT stay_id, observation_start_time AS window_start, observation_end_time AS window_end FROM read_csv_auto('{window_csv_path}')")
else:
    # icustays가 DB 내에 있다고 가정하고 생성
    con.execute("""
    CREATE OR REPLACE TABLE windows_1h AS
    WITH RECURSIVE time_array AS (
        SELECT stay_id, date_trunc('hour', intime) AS window_start, date_trunc('hour', intime) + INTERVAL '1 hour' AS window_end, outtime FROM icustays
        UNION ALL
        SELECT stay_id, window_start + INTERVAL '1 hour', window_end + INTERVAL '1 hour', outtime FROM time_array WHERE window_start + INTERVAL '1 hour' < outtime
    )
    SELECT stay_id, window_start, window_end FROM time_array;
    """)

In [13]:
# 3. 4종 승압제 통합 이벤트 뷰 생성 (Sparsity 및 다중공선성 고려 준비)
con.execute("""
CREATE OR REPLACE TEMP VIEW all_pressor_events AS
SELECT
    stay_id,
    CASE 
        WHEN itemid = 221906 THEN 'norepi'
        WHEN itemid = 222315 THEN 'vaso'
        WHEN itemid = 221289 THEN 'epi'
        WHEN itemid = 221662 THEN 'dopa'
    END AS drug_type,
    TRY_CAST(starttime AS TIMESTAMP) AS start_time,
    TRY_CAST(endtime AS TIMESTAMP) AS end_time
FROM inputevents
WHERE itemid IN (221906, 222315, 221289, 221662)
  AND starttime IS NOT NULL;
""")

In [14]:
# 4. 윈도우별 피처 계산 및 최종 코호트 테이블 생성
con.execute("""
CREATE OR REPLACE TABLE final_cohort_features AS
WITH window_agg AS (
    SELECT
        w.stay_id,
        w.window_start,
        w.window_end,
        -- 각 약물별 윈도우 내 투여 시간(0~1 사이) 계산
        SUM(CASE WHEN v.drug_type = 'norepi' THEN EXTRACT(EPOCH FROM LEAST(v.end_time, w.window_end) - GREATEST(v.start_time, w.window_start)) / 3600.0 ELSE 0 END) AS dur_norepi,
        SUM(CASE WHEN v.drug_type = 'vaso' THEN EXTRACT(EPOCH FROM LEAST(v.end_time, w.window_end) - GREATEST(v.start_time, w.window_start)) / 3600.0 ELSE 0 END) AS dur_vaso,
        SUM(CASE WHEN v.drug_type = 'epi' THEN EXTRACT(EPOCH FROM LEAST(v.end_time, w.window_end) - GREATEST(v.start_time, w.window_start)) / 3600.0 ELSE 0 END) AS dur_epi,
        SUM(CASE WHEN v.drug_type = 'dopa' THEN EXTRACT(EPOCH FROM LEAST(v.end_time, w.window_end) - GREATEST(v.start_time, w.window_start)) / 3600.0 ELSE 0 END) AS dur_dopa
    FROM windows_1h w
    LEFT JOIN all_pressor_events v ON v.stay_id = w.stay_id AND v.start_time < w.window_end AND v.end_time > w.window_start
    GROUP BY 1, 2, 3
)
SELECT 
    *,
    -- [다중공선성 대응 지표] 현재 동시 투여 중인 승압제 개수
    ((dur_norepi > 0)::INT + (dur_vaso > 0)::INT + (dur_epi > 0)::INT + (dur_dopa > 0)::INT) AS pressor_count
FROM window_agg;
""")

In [19]:
# 5. CSV 내보내기
output_path = "/home/oracle/Coding/wsl_projects/miniprj/data-pipeline/data/processed/final_pressor_cohort.csv"
df_final = con.execute("SELECT * FROM final_cohort_features ORDER BY stay_id, window_start").df()
df_final.to_csv(output_path, index=False)

print(f"코호트 테이블 생성 및 CSV 저장 완료: {output_path}")

코호트 테이블 생성 및 CSV 저장 완료: /home/oracle/Coding/wsl_projects/miniprj/data-pipeline/data/processed/final_pressor_cohort.csv
